# MaskGIT propio (ablación)
`python -m src.cli maskgit <acción>` en un notebook: cada sección es una acción de la CLI, en el mismo orden y con los mismos defaults, más una sección propia del modelo. MaskGIT con receta propia (el paper no publica el entrenamiento): decodificación paralela por confianza con calendario coseno. El MaskGIT fiel de video es `genie`; este es la ablación propia.

Guía completa: `Docs/modelos/maskgit.md`. Acciones de `maskgit`: `receta`, `entrenar`, `continuar`, `evaluar`, `bench`, `demo`, `exportar`.

**Cómo se usa.** Con `DRY_RUN = True` (lo de la primera celda) cada acción solo imprime los comandos que correría, igual que `--dry-run`. Con `DRY_RUN = False` corre de verdad por `Toolkit`, en segundo plano (el kernel sigue libre), con candado de GPU y cola, y queda en `results/bitacora.jsonl` con su línea de CLI exacta. Lo que todavía no existe (resultados, checkpoints) se muestra como *pendiente*.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

In [ ]:
MODELO = "maskgit"
TAMANO = "s"        # tamaño del modelo
RUN = "maskgit_s"       # nombre de la corrida: <modelo>_<tamaño>[_<variante>][_s<semilla>]
DRY_RUN = True      # True: solo imprime los comandos (= --dry-run). False: los corre de verdad, en segundo plano
DEVICE = nbui.device_inicial()      # el que detecta el repo: cuda, rocm, mps, cpu (dml y npu: con ONNX Runtime)
DEV = nbui.device_kw(DEVICE)        # = --device
tk = Toolkit(MODELO)
print(MODELO, "| acciones:", ", ".join(tk.acciones), "| corrida:", RUN, "| device:", DEVICE, "| dry-run:", DRY_RUN)

## 1. Receta y fidelidad
La receta de entrenamiento es la de la fuente del modelo (con lo forzado por datos o presupuesto marcado) y la tabla de fidelidad dice qué es fiel, qué lo fuerzan los datos o el presupuesto y qué es propio. CLI: `python -m src.cli maskgit receta`.

In [ ]:
r = nbui.receta(MODELO, TAMANO)        # la Recipe (optimizador, lr por tramos, wd, EMA, clip, batch, presupuesto) y lo forzado
nbui.fidelidad(MODELO);                # tabla de fidelidad: decisión, fuente, lo que hace el repo, categoría

In [ ]:
nbui.opciones(MODELO, "receta", tamano=TAMANO)   # opciones de la acción y el comando equivalente
t = nbui.lanzar(MODELO, "receta", dry_run=DRY_RUN, tamano=TAMANO)   # lo que imprime `python -m src.recipes`

## 2. Entrenar o continuar
Entrenar de cero con la receta de la fuente, probar el lr en un humo corto, seguir la corrida en vivo y continuarla o extenderla. CLI: `entrenar` (con `--humo`), `continuar`, `estado`.

In [ ]:
OPC_ENTRENAR = dict(tamano="s", limite=LIMITE)     # los mismos nombres que la CLI: --tamaño, --destilar, --variante, --limite...
nbui.opciones(MODELO, "entrenar", **OPC_ENTRENAR);

In [ ]:
# humo: 300 pasos con el lr de la fuente, escalado por raíz, lineal, x3 y x10; elige el que mejor baja (--humo)
t = nbui.lanzar(MODELO, "entrenar", dry_run=DRY_RUN, humo=True, **dict(tamano="s"))

In [ ]:
# entrenamiento de verdad (con DRY_RUN = False corre en segundo plano: el kernel sigue libre)
t = nbui.lanzar(MODELO, "entrenar", dry_run=DRY_RUN, **OPC_ENTRENAR)

In [ ]:
# seguimiento: barra con porcentaje y ETA, y curvas de loss, lr y monitoreo con el tramo del schedule sombreado
nbui.mostrar_barra(RUN)
nbui.curvas(RUN)
# nbui.en_vivo(RUN)     # se actualiza solo hasta que la corrida termina (en una corrida real)

In [ ]:
nbui.estado_corridas()                                 # una fila por corrida de checkpoints/ (= python -m src.status)
nbui.panel_cola();                                      # si la GPU estaba ocupada, la acción quedó en la cola

**Continuar.** Retomar una corrida cortada (`continuar --run`) o extenderla con más presupuesto (`--presupuesto 2x`): la extensión es una corrida nueva `<run>_x<k>` con `parent`, sha y receta. Las recetas con coseno solo se extienden con `--restart`.

In [ ]:
OPC_CONTINUAR = dict(run=RUN, limite=LIMITE)                       # retomar desde last.pt
OPC_EXTENDER = dict(run=RUN, presupuesto="2x", limite=LIMITE)      # extender: corrida nueva <run>_x2
nbui.opciones(MODELO, "continuar", **OPC_CONTINUAR)
t = nbui.lanzar(MODELO, "continuar", dry_run=DRY_RUN, **OPC_CONTINUAR)
t = nbui.lanzar(MODELO, "continuar", dry_run=DRY_RUN, **OPC_EXTENDER)

## 3. Evaluar
Métricas propias de video: CE, rollouts de 3 s, LPIPS y FVD con IC por segmento, y el piso de FVD (real contra real) al lado. CLI: `python -m src.cli maskgit evaluar`.

In [ ]:
OPC_EVALUAR = dict(run=RUN, pasos="8", limite=LIMITE, **DEV)
nbui.opciones(MODELO, "evaluar", **OPC_EVALUAR)
t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, **OPC_EVALUAR)

In [ ]:
nbui.tabla_video(solo=[RUN]);          # una fila: CE, LPIPS, FVD con IC por segmento, piso de FVD y latencia; "pendiente" sin JSON

Más opciones de `evaluar`: `--cadena` (calidad en línea y con la cadena completa), `--barrer-temperatura` (T elegida en el holdout), `--temperatura`, `--ctx-frames`, `--dtype`.

In [ ]:
t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, run=RUN, cadena=True, limite=LIMITE, **DEV)               # calidad con la cadena de punta a punta
t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, run=RUN, barrer_temperatura=True, limite=LIMITE, **DEV)    # barrido de T en el holdout

## 4. Latencia
Un protocolo único: un paso por frame en batch 1, fp16 con CUDA graphs donde anda, p50 y p99; con `--cadena` mide la cadena cámara → encoder → modelo → decoder, en bucle cerrado y a ritmo de cámara (50 ms). CLI: `python -m src.cli maskgit bench`.

In [ ]:
OPC_BENCH = dict(run=RUN, limite=LIMITE, **DEV)
nbui.opciones(MODELO, "bench", **OPC_BENCH)
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, **OPC_BENCH)                                   # el modelo solo
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, cadena=True, ritmo=50, **OPC_BENCH)            # la cadena a ritmo de cámara
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, cadena=True, diagnostico=True, **OPC_BENCH)    # el piso de ~20 ms a ritmo de cámara

La latencia no depende de los pesos: con `aleatorio=True` se mide la arquitectura sin entrenar.

In [ ]:
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, aleatorio=True, limite=LIMITE, **DEV)

In [ ]:
nbui.latencia_etapas();      # p50 y p99 de cada columna del protocolo y mediana por etapa, de results/infer/

## 5. Demo
Un mp4 con el video real a la izquierda y lo que predice el modelo a la derecha: 1 s de contexto real y después imagina. CLI: `python -m src.cli maskgit demo`.

In [ ]:
OPC_DEMO = dict(run=RUN, pasos="8", limite=LIMITE, **DEV)
nbui.opciones(MODELO, "demo", **OPC_DEMO)
# lo que va después de -- es de src.infer.demo: --ctx (frames de contexto), --horizon (frames imaginados), --segment, --video...
t = nbui.lanzar(MODELO, "demo", dry_run=DRY_RUN, extra=["--ctx", "20", "--horizon", "60"], **OPC_DEMO)
nbui.video(nbui.ruta_demo(RUN, pasos=8), "real (izquierda) contra predicho (derecha)");

## 6. Exportar
ONNX, donde aplica (`python -m src.cli maskgit exportar`). El export verifica que ONNX Runtime coincida con PyTorch.

In [ ]:
OPC_EXPORTAR = dict(run=RUN) if MODELO != "codec" else dict()
nbui.opciones(MODELO, "exportar", **OPC_EXPORTAR)
t = nbui.lanzar(MODELO, "exportar", dry_run=DRY_RUN, **OPC_EXPORTAR)
t = nbui.lanzar(MODELO, "exportar", dry_run=DRY_RUN, dtype="fp16", **OPC_EXPORTAR)      # la versión fp16

## 7. Sección propia: pasos de decodificación, choice_temp y cómo se destapa un frame
En cada paso el modelo muestrea todas las celdas tapadas, puntúa su confianza (log p más ruido Gumbel de `choice_temp·(1 − s/S)`) y fija las más confiables hasta dejar tapadas `floor(128·cos(π/2·s/S))` celdas. Con un paso es argmax; con más pasos el frame se destapa de a poco.

In [ ]:
nbui.calendario_destapado((1, 2, 4, 8, 12));      # cuántas celdas siguen tapadas tras cada paso

**Un frame, paso a paso.** Es el `decode_frame` real del modelo sobre un modelo de juguete (logits sintéticos con confianza distinta por celda): oscuro = ya fijado. Con `choice_temp` alto el orden se mezcla al principio; con 0 se fijan siempre las más confiables.

In [ ]:
for ct in (0.0, 4.5):
  nbui.destapar_frame(pasos=8, choice_temp=ct, temperatura=1.0)

**Barrido de pasos** (1, 2, 4, 8, 12): cuesta un paso del modelo por paso de decodificación, así que la latencia crece con S mientras el FVD mejora hasta cierto punto.

In [ ]:
t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, run=RUN, pasos="1,2,4,8,12", limite=LIMITE, **DEV)
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, run=RUN, pasos="1,2,4,8,12", limite=LIMITE, **DEV)
nbui.curva_pasos(RUN);

**Temperatura.** `--temperatura` cambia el muestreo de cada celda y queda en el nombre del resultado (`_t<T>`). El barrido de `choice_temp` sobre el modelo entrenado necesita una opción propia del Toolkit (hoy solo se puede pedir con `--opt choice_temp=...` por `--`, que pisaría el resultado de la misma corrida): acá se lo ve en el destapado.

In [ ]:
for T in (0.8, 1.0):
  t = nbui.lanzar(MODELO, "evaluar", dry_run=DRY_RUN, run=RUN, pasos="8", temperatura=T, limite=LIMITE, **DEV)